# DFA Operations

This notebook studies operations on deterministic finite automata (DFAs). In automata theory, we often build new automata from existing ones to recognize languages formed by combining simpler languages.

We will focus on the most important DFA operations:

- union
- intersection
- complement
- difference

The main idea is that if two DFAs recognize languages L1 and L2, then we can construct a DFA for L1 ∪ L2, L1 ∩ L2, complement of L1, and L1 \ L2 using product construction and state labeling.


## 👨‍💻 Author

### **Muhammad Ali**

[![GitHub](https://img.shields.io/badge/GitHub-AliMuhammad78-181717?style=flat-square\&logo=github\&logoColor=white)](https://github.com/AliMuhammad78)
[![LinkedIn](https://img.shields.io/badge/LinkedIn-Muhammad%20Ali-0A66C2?style=flat-square\&logo=linkedin\&logoColor=white)](https://www.linkedin.com/in/muhammad-ali-91294a290)
[![Kaggle](https://img.shields.io/badge/Kaggle-ali98muhammad45-20BEFF?style=flat-square\&logo=kaggle\&logoColor=white)](https://www.kaggle.com/ali98muhammad45)

## Learning objectives

By the end of this notebook, you should be able to:

- explain what it means to combine languages through set operations,
- describe how DFA operations are implemented using product states,
- construct a DFA for the union, intersection, complement, and difference of two languages,
- reason about acceptance and rejection in product automata,
- implement these operations in Python and test them on small examples.

## Basic theory

A language is a set of strings. If a DFA recognizes a language L, then we can combine two DFAs to recognize new languages formed by standard set operations.

For two languages L1 and L2:

- Union: L1 ∪ L2 = {w | w ∈ L1 or w ∈ L2}
- Intersection: L1 ∩ L2 = {w | w ∈ L1 and w ∈ L2}
- Complement: L1^c = {w | w ∉ L1}
- Difference: L1 - L2 = {w | w ∈ L1 and w ∉ L2}

Because DFAs are closed under these operations, we can build new DFAs that accept exactly those languages.

## Important idea: product construction

Suppose we have two DFAs:

- M1 = (Q1, Σ, δ1, q01, F1)
- M2 = (Q2, Σ, δ2, q02, F2)

We can simulate both machines in parallel. Each combined state is a pair (p, q) where p is a state in M1 and q is a state in M2.

For every input symbol a, we move to:

(δ1(p, a), δ2(q, a))

This is called the product construction.

The acceptance condition depends on the operation:

- Union: accept if either machine accepts
- Intersection: accept if both machines accept
- Difference: accept if M1 accepts and M2 rejects

## Terminology and notation

- DFA: deterministic finite automaton
- Language: a set of strings
- State: a machine configuration
- Start state: initial state before reading input
- Accepting state: a state that accepts the current input
- Product state: a pair of states from two DFAs
- Complement: all strings not accepted by the automaton
- Difference: strings accepted by one DFA but rejected by another

## Why these operations work

A DFA determines acceptance by the final state reached after reading the whole input. In the product automaton, we track the current state of both DFAs at the same time.

That means we can decide acceptance based on both machines simultaneously:

- union accepts if either final state is accepting,
- intersection accepts if both are accepting,
- complement flips acceptance,
- difference accepts if the first machine accepts and the second does not.

This is a fundamental and elegant property of regular languages.

## Simple example: two small DFAs

Let Σ = {a, b}. Consider two DFAs:

- M1 accepts strings that contain at least one a
- M2 accepts strings that end with b

We can combine them to build a new DFA for:

- M1 ∪ M2
- M1 ∩ M2
- M1 - M2

The product construction is exactly how we do this.


In [ ]:
# A small DFA representation
# Each state maps an input symbol to the next state.

M1 = {
    'q0': {'a': 'q1', 'b': 'q0'},
    'q1': {'a': 'q1', 'b': 'q1'}
}
M1_accepting = {'q1'}
M1_start = 'q0'

M2 = {
    'r0': {'a': 'r0', 'b': 'r1'},
    'r1': {'a': 'r0', 'b': 'r1'}
}
M2_accepting = {'r1'}
M2_start = 'r0'

print("M1 accepts strings with at least one a.")
print("M2 accepts strings ending with b.")
print("Example states: M1 start =", M1_start, "and M2 start =", M2_start)


## Example 1: union

The union of two languages L1 and L2 contains all strings that belong to either language.

If M1 accepts L1 and M2 accepts L2, then in the product automaton we accept whenever either DFA is in an accepting state.

This means the union DFA accepts if:

(p is accepting) OR (q is accepting)

## Example 2: intersection

The intersection contains exactly those strings that belong to both languages.

Thus, the intersection DFA accepts if:

(p is accepting) AND (q is accepting)

This is a standard product-state test.

## Example 3: complement

The complement of a language L is all strings not in L.

If a DFA accepts L, its complement is obtained by flipping accepting and nonaccepting states.

So for each state p, we accept if p is not accepting in the original DFA.

## Example 4: difference

The difference L1 - L2 contains strings in L1 that are not in L2.

That means the combined DFA accepts if:

(p is accepting) AND (q is not accepting)

## Python implementation: product construction

The following function builds the product of two DFAs. After that, we can define operations by selecting the right acceptance rule.


In [ ]:
# Product construction for DFA operations

from collections import deque


def product_dfa(dfa1, start1, accept1, dfa2, start2, accept2, alphabet):
    """Build the product DFA of two DFAs."""
    queue = deque([(start1, start2)])
    visited = set()
    product = {}

    while queue:
        state = queue.popleft()
        if state in visited:
            continue
        visited.add(state)

        p, q = state
        product[state] = {}

        for symbol in alphabet:
            next1 = dfa1[p][symbol]
            next2 = dfa2[q][symbol]
            next_state = (next1, next2)
            product[state][symbol] = next_state

            if next_state not in visited:
                queue.append(next_state)

    return product


def run_dfa(dfa, start, accepting, input_string):
    """Check whether a DFA accepts a string."""
    state = start
    for symbol in input_string:
        state = dfa[state][symbol]
    return state in accepting


alphabet = {'a', 'b'}

# Example 1: union
union_product = product_dfa(M1, M1_start, M1_accepting, M2, M2_start, M2_accepting, alphabet)

# Accept if either machine accepts.
def union_accepts(state, accept1, accept2):
    p, q = state
    return p in accept1 or q in accept2

# Example 2: intersection
# Accept if both machines accept.
def intersection_accepts(state, accept1, accept2):
    p, q = state
    return p in accept1 and q in accept2

# Example 3: complement
# Accept if the first machine does not accept.
def complement_accepts(state, accept1):
    p, q = state
    return p not in accept1

# Example 4: difference
# Accept if the first accepts and the second rejects.
def difference_accepts(state, accept1, accept2):
    p, q = state
    return p in accept1 and q not in accept2

# Example strings
for word in ['', 'a', 'b', 'ab', 'bb', 'aaab']:
    print("Word:", word)
    print("  M1:", run_dfa(M1, M1_start, M1_accepting, word))
    print("  M2:", run_dfa(M2, M2_start, M2_accepting, word))
    print("  Union:", union_accepts((M1_start, M2_start), M1_accepting, M2_accepting))
    print("  Intersection:", intersection_accepts((M1_start, M2_start), M1_accepting, M2_accepting))

print("\nThe product construction allows us to test the combined behavior of both DFAs at once.")


## Practice exercises

### Exercise 1: easy

Let L1 be the language of strings containing at least one a, and L2 be the language of strings ending in b.

Construct the DFA for:

- L1 ∪ L2
- L1 ∩ L2
- L1 - L2

Explain which strings are accepted by each.

### Exercise 2: medium

Suppose we have two DFAs over Σ = {0,1}.

- M1 accepts strings with an even number of 1s.
- M2 accepts strings that end with 0.

Build the product construction and describe the accepting states for:

- M1 ∩ M2
- M1 ∪ M2
- complement of M1

### Exercise 3: challenging

Create two DFA examples where the difference language L1 - L2 is nonempty, but the intersection is empty. Explain how the product automaton distinguishes the two behaviors.

### Exercise 4: implementation challenge

Write a function that, given two DFAs, returns the DFA for the difference language L1 - L2. Test it on several sample strings and verify the outputs by hand.

## Final summary

DFA operations are built from the idea of running two machines in parallel. The product construction is the main method used to implement union, intersection, complement, and difference. Each operation changes only the acceptance condition, while the underlying combined state space remains the same.

This is an important result in automata theory because regular languages are closed under these operations.


## Summary

In this notebook, we learned that:

- DFAs can be combined using the product construction,
- union and intersection are implemented by checking accepting states in the product,
- complement is obtained by flipping accepting and nonaccepting states,
- difference is accepted when the first DFA accepts and the second rejects,
- regular languages are closed under these operations.

These operations are fundamental because they let us build new automata to describe more complicated languages from simpler ones.
